# EXO_M2_F05_PRODUCTION — Batch Alchemist Mode 2

```
╔══════════════════════════════════════════════════════════════════════════════╗
║    MODE 2 — M2_F05 ALCHEMIST — PRODUCTION — BATCH FUSION VISUELLE           ║
║                                                                              ║
║   Setup → Frames M2_F04 → Source Vidéo (opt.) → Batch → OUT_FINAL_FRAMES/  ║
║                                                                              ║
║   Pipeline : match_color → grain → bloom → sharpness (CPU OpenCV)            ║
║   CLI      : EXO_M2_F05_ALCHEMIST.py --preset --production-plan             ║
║   Output   : PNG 16-bit dans OUT_FINAL_FRAMES/                              ║
║   Loi R-01 : Copie étanche Mode 2 — zéro contamination Mode 1               ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

**Mode:** Exécution batch après validation CONTROL. 4 cellules.

In [ ]:
#@title 🔗 [EXODUS] Drive + Session JSON
#@markdown Monte le Drive et lit exodus_session.json genere par EXO_LAUNCHER
from google.colab import drive
drive.mount('/content/drive')

import sys, json
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/EXODUS_V2"  #@param {type:"string"}
sys.path.insert(0, DRIVE_ROOT)

_session_path = Path(DRIVE_ROOT) / "exodus_session.json"
if _session_path.exists():
    with open(_session_path) as _f:
        EXODUS_SESSION = json.load(_f)
    print("OK exodus_session.json charge")
    print(f"  Mode     : {EXODUS_SESSION['mode']} --- {EXODUS_SESSION['mode_label']}")
    print(f"  Timestamp: {EXODUS_SESSION['timestamp']}")
    print(f"  Drive    : {EXODUS_SESSION['drive_root']}")
else:
    print("ATTENTION : exodus_session.json introuvable")
    print("   -> Lancer EXO_LAUNCHER.ipynb d'abord.")
    EXODUS_SESSION = {
        "mode": None, "mode_label": "UNKNOWN",
        "drive_root": DRIVE_ROOT, "status": "missing"
    }

## 1. Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path("/content/drive/MyDrive/EXODUS_V2")
# DRIVE_ROOT = Path("/home/EXODUS-V2")  # Local dev

FREGATE_ROOT = DRIVE_ROOT / "11_M2_F05_ALCHEMIST"
CODEBASE     = FREGATE_ROOT / "CODEBASE"
IN_RAW_FRAMES    = FREGATE_ROOT / "IN_RAW_FRAMES"
IN_SOURCE_REF    = FREGATE_ROOT / "IN_SOURCE_REF"
IN_PLAN          = FREGATE_ROOT / "IN_PRODUCTION_PLAN"
OUT_FINAL_FRAMES = FREGATE_ROOT / "OUT_FINAL_FRAMES"

sys.path.insert(0, str(CODEBASE))

!pip install -q numpy opencv-python-headless Pillow tqdm

PRESET    = "cinema_fusion"
PLAN_PATH = IN_PLAN / "PRODUCTION_PLAN.JSON"

print(f"Drive Root       : {DRIVE_ROOT}")
print(f"Frégate Root     : {FREGATE_ROOT}")
print(f"IN_RAW_FRAMES    : {IN_RAW_FRAMES}")
print(f"IN_SOURCE_REF    : {IN_SOURCE_REF}")
print(f"OUT_FINAL_FRAMES : {OUT_FINAL_FRAMES}")
print(f"Preset           : {PRESET}")
print(f"Plan             : {PLAN_PATH} — exists: {PLAN_PATH.exists()}")

## ⚡ GPU Check

In [ ]:
import subprocess

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                             '--format=csv,noheader'],
                            capture_output=True, text=True, timeout=10)
    if result.returncode == 0 and result.stdout.strip():
        parts = result.stdout.strip().split(',')
        print(f'[GPU] {parts[0].strip()} — {parts[1].strip()} — Driver {parts[2].strip()}')
    else:
        print('[GPU] Aucun GPU détecté — mode CPU (normal pour Alchemist)')
except Exception as e:
    print(f'[GPU] nvidia-smi non disponible — mode CPU')

## 2. Dry-run (validation)

In [ ]:
# Valider avant production
!python {CODEBASE}/EXO_M2_F05_ALCHEMIST.py \
  --production-plan {PLAN_PATH} \
  --preset {PRESET} \
  --dry-run --verbose

## 3. Production

In [ ]:
# Détecter automatiquement la vidéo source dans IN_SOURCE_REF/
source_videos = list(IN_SOURCE_REF.glob('*.mp4')) + list(IN_SOURCE_REF.glob('*.mov'))
SOURCE_VIDEO_ARG = f'--source-video {source_videos[0]}' if source_videos else ''
print(f'Source vidéo : {source_videos[0] if source_videos else "(aucune — bloom seul)"}')

!python {CODEBASE}/EXO_M2_F05_ALCHEMIST.py \
  --production-plan {PLAN_PATH} \
  --preset {PRESET} \
  {SOURCE_VIDEO_ARG} \
  --verbose

## 4. Vérification output

In [ ]:
import json

report_path = FREGATE_ROOT / "OUT_REPORT" / "m2_f05_report.json"
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    print(json.dumps(report.get('summary', {}), indent=2))
else:
    print('Rapport non trouvé')

frames = list(OUT_FINAL_FRAMES.glob('*.png'))
print(f'Frames output : {len(frames)} PNG dans {OUT_FINAL_FRAMES}')